## Setup ChromaDB

In [1]:
import json
import os

# Load all our data
with open("data/cleaned_text/all_cleaned.json", 
          encoding="utf-8") as f:
    cleaned_docs = json.load(f)

with open("data/entities/combined_entities.json",
          encoding="utf-8") as f:
    entities = json.load(f)

print("✓ Data loaded")
print(f"Documents: {len(cleaned_docs)}")

# Setup ChromaDB
import chromadb

client = chromadb.PersistentClient(path="data/chromadb")
print("✓ ChromaDB ready")

✓ Data loaded
Documents: 3
✓ ChromaDB ready


In [2]:
from sentence_transformers import SentenceTransformer

print("Loading multilingual sentence transformer...")
print("This handles both English and Arabic...")

model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

print("✓ Multilingual model loaded!")

# Test it
test_texts = [
    "limestone statue found near pyramid",
    "وجدنا تمثال من الحجر الجيري",
]

embeddings = model.encode(test_texts)
print(f"✓ Embedding shape: {embeddings.shape}")
print("Ready to embed Arabic and English together!")

/Users/juhi/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading multilingual sentence transformer...
This handles both English and Arabic...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Multilingual model loaded!
✓ Embedding shape: (2, 384)
Ready to embed Arabic and English together!


In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# Create fresh collection
try:
    client.delete_collection("arabic_documents")
except:
    pass

collection = client.create_collection(
    name="arabic_documents",
    metadata={"description": "Arabic historical documents"}
)

print("Indexing documents into ChromaDB...")

documents = []
metadatas = []
ids = []
embeddings_list = []

for doc_name, doc_data in cleaned_docs.items():
    text = doc_data['cleaned_text']
    
    doc_entities = entities.get(doc_name, {}).get('entities', {})
    
    persons = ', '.join(doc_entities.get('PERS', []) + 
                       doc_entities.get('PERSON', []))
    locations = ', '.join(doc_entities.get('LOC', []) + 
                         doc_entities.get('LOCATION', []) +
                         doc_entities.get('PLACE', []))
    orgs = ', '.join(doc_entities.get('ORG', []) + 
                    doc_entities.get('ORGANIZATION', []))
    
    embedding = model.encode(text).tolist()
    
    documents.append(text)
    metadatas.append({
        "source": doc_name,
        "description": doc_data['description'],
        "persons": persons,
        "locations": locations,
        "organizations": orgs,
        "char_count": str(doc_data['clean_chars'])
    })
    ids.append(doc_name.replace('.jpg', '').replace('.', '_'))
    embeddings_list.append(embedding)
    
    print(f"✓ Indexed: {doc_name}")
    print(f"  Persons:   {persons[:60] if persons else 'none'}")
    print(f"  Locations: {locations[:60] if locations else 'none'}")

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)

print(f"\n✓ All {len(documents)} documents indexed!")
print(f"✓ Collection size: {collection.count()}")

Indexing documents into ChromaDB...
✓ Indexed: arabic_doc_001.jpg
  Persons:   ايهالقا بو, بن فوا
  Locations: لسمياق, ااا ادس, ستعسين, سناغا, ارلا
✓ Indexed: arabic_doc_004.jpg
  Persons:   يي, ريمت بن, معدو, سوك بن ماش, ايريصي سين, معن بن ميلك, بن م
  Locations: البمنية
✓ Indexed: arabic_doc_005.jpg
  Persons:   none
  Locations: والناط, عاراذ, اباصاحة, بقن خزراذالويزبزج, ابرزباف, بت, القن

✓ All 3 documents indexed!
✓ Collection size: 3


In [5]:
def search_documents(query, n_results=3):
    """
    Search Arabic historical documents using any language query.
    Fixed similarity score calculation.
    """
    print(f"\nSEARCH QUERY: '{query}'")
    print("="*60)
    
    query_embedding = model.encode(query).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(n_results, collection.count())
    )
    
    distances = results['distances'][0]
    
    # Find min and max for normalization
    min_d = min(distances)
    max_d = max(distances)
    
    for i, (doc_id, document, metadata, distance) in enumerate(zip(
        results['ids'][0],
        results['documents'][0],
        results['metadatas'][0],
        distances
    )):
        # Normalize: closest document = 100%, farthest = 0%
        if max_d != min_d:
            similarity = round(
                (1 - (distance - min_d) / (max_d - min_d)) * 100, 1)
        else:
            similarity = 100.0
        
        rank_label = ["🥇", "🥈", "🥉"][i]
        
        print(f"\n{rank_label} RESULT {i+1}: {metadata['source']}")
        print(f"   Relevance: {similarity}%")
        print(f"   Type: {metadata['description']}")
        if metadata['persons']:
            print(f"   👤 People: {metadata['persons'][:80]}")
        if metadata['locations']:
            print(f"   📍 Places: {metadata['locations'][:60]}")
        if metadata['organizations']:
            print(f"   🏛 Orgs: {metadata['organizations'][:60]}")
        print(f"   📄 Text: {document[:100]}...")
    
    return results

# Test searches
print("="*60)
print("SEMANTIC SEARCH: Arabic Documents")
print("="*60)

queries = [
    "names of people and family records",
    "official government administrative document", 
    "historical manuscript with dense text",
    "أسماء الأشخاص",
    "date and location",
]

for q in queries:
    search_documents(q)
    print()

SEMANTIC SEARCH: Arabic Documents

SEARCH QUERY: 'names of people and family records'

🥇 RESULT 1: arabic_doc_001.jpg
   Relevance: 100.0%
   Type: Two page manuscript black and red ink
   👤 People: ايهالقا بو, بن فوا
   📍 Places: لسمياق, ااا ادس, ستعسين, سناغا, ارلا
   🏛 Orgs: ايناد
   📄 Text: سس سات لخر ليجع ويه ستعسين ١ زوه اليه ااا ادس سناغا درس ارلا الور لط
لين ننه الذي اطل مس انور الس ال...

🥈 RESULT 2: arabic_doc_005.jpg
   Relevance: 52.0%
   Type: Dense Arabic prose manuscript
   📍 Places: والناط, عاراذ, اباصاحة, بقن خزراذالويزبزج, ابرزباف, بت, القن
   🏛 Orgs: غورنالتنطزعل, الاو زيرف, نادي مال, لمر ريجياه, اشنا, والشتاز
   📄 Text: اباصاحة السمل المروفة بقن خزراذالويزبزج والناط يفل القنظوض ٠.
ابالرسا ركان بت عاراذ ابرزباف لقيةاوز ...

🥉 RESULT 3: arabic_doc_004.jpg
   Relevance: 0.0%
   Type: Handwritten administrative document with names and dates
   👤 People: يي, ريمت بن, معدو, سوك بن ماش, ايريصي سين, معن بن ميلك, بن ميلك, بن ماش
   📍 Places: البمنية
   🏛 Orgs: محافظة شهرة ال

In [6]:
print("="*60)
print("PIPELINE STATUS REPORT")
print("="*60)

import os
import json

# Count everything we built
with open("data/ocr_output/all_ocr_results.json") as f:
    ocr = json.load(f)

with open("data/entities/combined_entities.json") as f:
    ents = json.load(f)

total_chars = sum(d['char_count'] for d in ocr.values())
total_entities = sum(d['total'] for d in ents.values())

print(f"📄 Documents processed: {len(ocr)}")
print(f"📝 Total characters extracted: {total_chars}")
print(f"🏷 Total entities found: {total_entities}")
print(f"🔍 Search collection size: {collection.count()}")
print()
print("Entity breakdown:")
for doc, data in ents.items():
    print(f"\n  {doc}:")
    for etype, elist in data['entities'].items():
        if elist:
            print(f"    {etype}: {len(elist)} entities")

print()
print("✓ Pipeline complete through semantic search!")
print("✓ Ready for knowledge graph and web interface!")

PIPELINE STATUS REPORT
📄 Documents processed: 3
📝 Total characters extracted: 2373
🏷 Total entities found: 36
🔍 Search collection size: 3

Entity breakdown:

  arabic_doc_001.jpg:
    LOC: 5 entities
    PERS: 1 entities
    ORG: 1 entities
    MISC: 1 entities
    PERSON: 1 entities

  arabic_doc_004.jpg:
    PERS: 6 entities
    ORG: 2 entities
    PERSON: 2 entities
    PLACE: 1 entities

  arabic_doc_005.jpg:
    LOC: 8 entities
    ORG: 8 entities

✓ Pipeline complete through semantic search!
✓ Ready for knowledge graph and web interface!
